# 06_03 Activations and loss: how a network bends, and how it knows it is wrong

The bend comes from the activation function, and the network's sense of being wrong comes from the loss
function. This notebook draws the activations the book lists, asks PyTorch for their slopes, and computes
the loss a classifier reports before it has learned anything, which turns out to be a number you can
predict exactly.

**How this notebook works.** Every notebook in this course has the same rhythm:

1. **Recall.** Answer from memory before you look anything up. `ask()` tells you at once whether you were right.
2. **Predict, then run.** Before a cell with a surprise in it, write your prediction into `guess()`. The next cell runs the code and `reveal()` compares.
3. **Worked example, then your turn.** One example is done in full; the next, near-identical one has lines marked `# YOUR CODE HERE`.
4. **Check.** A `check_...()` cell tests what you saved, exactly as the checkpoint will, and says what to fix.

Run cells in order with **Shift+Enter**. If you get lost, **Kernel, Restart Kernel and Run All Cells** starts clean.

Running this in Google Colab? This cell sets it up; in CourseLabs it does nothing.

In [ ]:
# Colab setup. In a CourseLabs session this cell does nothing.
import os, sys
if "google.colab" in sys.modules:
    import importlib, importlib.util, subprocess
    LAB, REPO = "lab-nlp-06-a-neuron-from-scratch", "/content/nlp-course"
    if not os.path.isdir(REPO):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/fenago/nlp-course.git", REPO], check=True)
    os.chdir(f"{REPO}/{LAB}")
    if not os.path.exists("data"):
        os.symlink("../data", "data")
    os.makedirs("out", exist_ok=True)
    os.environ["NLPLAB_DATA"] = f"{REPO}/data"
    sys.path.insert(0, os.getcwd())
    PIP = {'sentence_transformers': 'sentence-transformers',
           'torch': 'torch',
           'sklearn': 'scikit-learn',
           'pandas': 'pandas',
           'numpy': 'numpy',
           'matplotlib': 'matplotlib'}
    missing = [spec for mod, spec in PIP.items() if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
        importlib.invalidate_caches()
    print(f"Ready: {LAB} and its data are in {os.getcwd()}; installed {len(missing)} package(s).")
elif not os.path.isdir("/opt/nlplab/data") and os.path.isdir("data"):
    # A downloaded copy on your own computer: the helpers read data/ from here.
    os.environ["NLPLAB_DATA"] = os.path.abspath("data")

In [ ]:
import json
import math
import os
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from nlpcheck import ask, guess, reveal, check_06_03

## 1. Recall

**r5.** Three linear layers with no activation between them can compute exactly what how many linear
layers can? (a number)

**r6.** If every weight in a network starts at the same value, its hidden units... (a) learn different
features, (b) stay identical to each other, (c) become random

In [ ]:
ask("r5", "")
ask("r6", "")

## 2. The activations, and their slopes

Each function below takes a neuron's weighted sum and bends it. **GELU**, the last one, is a smooth version
of ReLU and is the activation inside BERT and most current language models.

In [ ]:
z = torch.linspace(-5, 5, 201)
fns = {"step": lambda v: (v > 0).float(), "sigmoid": torch.sigmoid, "tanh": torch.tanh,
       "ReLU": torch.relu, "Leaky ReLU": nn.LeakyReLU(0.01), "GELU": nn.GELU()}
fig, axes = plt.subplots(2, 3, figsize=(12, 6))
for ax, (name, f) in zip(axes.flat, fns.items()):
    ax.plot(z, f(z)); ax.axhline(0, lw=0.5, c="k"); ax.axvline(0, lw=0.5, c="k"); ax.set_title(name)
plt.tight_layout(); plt.show()

Training moves each weight in proportion to the **slope** (gradient) of everything after it, so the slope of
the activation matters as much as its shape. PyTorch computes slopes for you: mark a tensor with
`requires_grad=True`, compute, call `.backward()`, and read `.grad`.

Predict the sigmoid's slope at 3, to two decimal places. (Its steepest slope, at 0, is 0.25.)

In [ ]:
guess("sigmoid_slope_at_3", None)

In [ ]:
for name, f in fns.items():
    if name == "step":
        continue
    for at in (-3.0, 0.0, 3.0, 6.0):
        v = torch.tensor(at, requires_grad=True)
        f(v).backward()
        print(f"{name:10} slope at {at:+.0f}: {v.grad.item():.4f}")
    print()
v = torch.tensor(3.0, requires_grad=True); torch.sigmoid(v).backward()
reveal("sigmoid_slope_at_3", round(v.grad.item(), 2))

0.05: a fifth of its best, and at 6 it is 0.0025. A network of sigmoids multiplies those small slopes layer
after layer, so the early layers barely learn: the **vanishing gradient**. ReLU's slope is exactly 1 for any
positive input, which is why it replaced the sigmoid in hidden layers, and exactly 0 for any negative input,
which is how a ReLU neuron can switch off for good (**dying ReLU**). Leaky ReLU keeps 0.01 there. The step
function has no slope at all almost everywhere, which is why nothing can be trained through it.

## 3. Softmax: from scores to probabilities

A classifier with several classes ends with one score per class. **Softmax** turns them into probabilities:
it exponentiates each score and divides by the total. The book's example:

In [ ]:
scores = torch.tensor([1.2, 0.9, 0.75])
p = torch.softmax(scores, dim=0)
print(p, "sum", p.sum().item())

0.42, 0.31 and 0.27, summing to 1. The largest score wins, but not by as much as you might guess from the
scores: softmax keeps the model's uncertainty visible.

## 4. Loss for numbers: which one does one outlier hurt most?

Five predictions of five values, all close except the last, which is off by 10. Predict which loss grows the
most in proportion when that one bad prediction is added: `mae`, `mse` or `huber`.

In [ ]:
y     = torch.tensor([3., 5., 2., 7., 4.])
good  = torch.tensor([2.5, 5.5, 2., 6.5, 4.5])
worse = torch.tensor([2.5, 5.5, 2., 6.5, 14.])
guess("outlier_loss", None)   # "mae", "mse" or "huber" 

In [ ]:
losses = {"mae": nn.L1Loss(), "mse": nn.MSELoss(), "huber": nn.HuberLoss(delta=1.0)}
growth = {}
for name, f in losses.items():
    a, b = f(good, y).item(), f(worse, y).item()
    growth[name] = b / a
    print(f"{name:5}  without the outlier {a:.3f}   with it {b:.3f}   {b / a:.0f} times larger")
print("rmse  ", round(nn.MSELoss()(worse, y).sqrt().item(), 3))
reveal("outlier_loss", max(growth, key=growth.get))

Mean squared error grows about a hundredfold, because it squares the error: an error of 10 counts as 100.
Mean absolute error grows about sixfold, because the outlier counts only by its size. Huber loss, about
twentyfold, sits between them: it is squared for small errors, so it learns precisely from the good
predictions, and absolute for large ones, so the outlier counts by its size rather than its square. None of
them is right in general: squared error suits a problem where big errors really are worse, absolute error
one where outliers are noise.

## 5. Loss for classes: cross-entropy, by hand

For classification the loss is **cross-entropy**: minus the log of the probability the model gave the right
class. A confident right answer costs nearly 0; a confident wrong answer costs a lot. Predict the loss of an
**untrained** network with 4 classes, like the ticket router of Part 3, to two decimal places.

In [ ]:
guess("untrained_loss", None)

In [ ]:
torch.manual_seed(0)
net = nn.Sequential(nn.Linear(384, 64), nn.ReLU(), nn.Linear(64, 4))
fake_inputs = torch.randn(600, 384) / 20          # the size of the ticket embeddings of Part 3
fake_labels = torch.randint(0, 4, (600,))
untrained = nn.CrossEntropyLoss()(net(fake_inputs), fake_labels).item()
print("untrained loss", round(untrained, 4), "   ln 4 =", round(math.log(4), 4))
reveal("untrained_loss", round(untrained, 2))

1.39, which is ln 4. An untrained network spreads its probability roughly evenly, a quarter per class, and
-ln(1/4) = ln 4. This is worth remembering: the first loss a classifier reports should be about ln(number of
classes). If it is far from that, the labels, the loss or the network's last layer is wrong, and you have
found the bug before training.

**Your turn.** Compute the cross-entropy of these three examples by hand, from the definition, and compare
it with PyTorch's. For each row: softmax the scores, take the probability of the right class, take minus its
log; then average the three.

In [ ]:
logits = torch.tensor([[2.0, 1.0, 0.1, -1.0], [0.5, 2.5, 0.3, 0.0], [1.2, 0.9, 0.75, 0.2]])
labels = torch.tensor([0, 1, 3])

by_hand = None   # YOUR CODE HERE: softmax each row, pick the right class, -log, average
from_torch = nn.CrossEntropyLoss()(logits, labels).item()
print("by hand", by_hand, "   PyTorch", from_torch)

In [ ]:
os.makedirs("out", exist_ok=True)
json.dump({"by_hand": None if by_hand is None else float(by_hand), "torch": from_torch, "untrained": untrained},
          open("out/06_03_losses.json", "w"), indent=1)
check_06_03()

## 6. Exit ticket

**x2.** Which activation keeps a small slope for negative inputs, so neurons cannot switch off for good?
(a) ReLU, (b) sigmoid, (c) Leaky ReLU

**x3.** Before any training, a 4-class classifier's cross-entropy loss is about: (a) 0, (b) 1.39, the natural
log of 4, (c) 4

In [ ]:
ask("x2", "")
ask("x3", "")